# Aerial Cactus Identification (YOLO26m-cls)Notebook-first refactor for Kaggle Aerial Cactus Identification with real competition download, strict dataset verification, YOLO26m-cls fine-tuning, and honest evaluation outputs.

## Dataset SourceCompetition link: https://www.kaggle.com/competitions/aerial-cactus-identification/dataThis notebook downloads competition files in code, verifies expected files and labels, creates train/val splits, and evaluates classification results on a held-out validation split.

In [ ]:
import importlibimport subprocessimport sysdef ensure_package(import_name, pip_name=None):    package_name = pip_name or import_name    try:        importlib.import_module(import_name)    except ImportError:        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package_name])ensure_package('numpy')ensure_package('pandas')ensure_package('matplotlib')ensure_package('sklearn', 'scikit-learn')ensure_package('PIL', 'Pillow')ensure_package('yaml', 'pyyaml')ensure_package('ultralytics')ensure_package('kaggle')print('Dependencies are ready.')

In [ ]:
import jsonimport osimport randomimport shutilimport zipfilefrom pathlib import Pathimport matplotlib.pyplot as pltimport numpy as npimport pandas as pdfrom PIL import Imagefrom IPython.display import displayfrom sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrixfrom sklearn.model_selection import train_test_splitSEED = 42random.seed(SEED)np.random.seed(SEED)PROJECT_DIR = Path.cwd()WORK_DIR = PROJECT_DIR / 'working'RAW_DIR = WORK_DIR / 'raw_kaggle'DATA_DIR = WORK_DIR / 'prepared_cls'ARTIFACTS_DIR = PROJECT_DIR / 'artifacts'WORK_DIR.mkdir(parents=True, exist_ok=True)RAW_DIR.mkdir(parents=True, exist_ok=True)DATA_DIR.mkdir(parents=True, exist_ok=True)ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)print(f'Project dir: {PROJECT_DIR}')

## Real Competition Download

In [ ]:
if not os.getenv('KAGGLE_USERNAME') or not os.getenv('KAGGLE_KEY'):    raise EnvironmentError('Missing Kaggle credentials. Set KAGGLE_USERNAME and KAGGLE_KEY.')download_cmd = [    sys.executable, '-m', 'kaggle', 'competitions', 'download',    '-c', 'aerial-cactus-identification',    '-p', str(RAW_DIR)]subprocess.check_call(download_cmd)zip_files = list(RAW_DIR.glob('*.zip'))if len(zip_files) == 0:    raise RuntimeError('No zip files downloaded from Kaggle competition.')for zf in zip_files:    with zipfile.ZipFile(zf, 'r') as z:        z.extractall(RAW_DIR)print('Downloaded and extracted competition files.')print(f'Raw dir: {RAW_DIR}')

## Dataset Verification: Files, Labels, and Splits

In [ ]:
train_csv = RAW_DIR / 'train.csv'train_img_dir = RAW_DIR / 'train'test_img_dir = RAW_DIR / 'test'if not train_csv.exists():    raise RuntimeError('train.csv missing after extraction.')if not train_img_dir.exists():    raise RuntimeError('train image directory missing after extraction.')if not test_img_dir.exists():    raise RuntimeError('test image directory missing after extraction.')df = pd.read_csv(train_csv)if 'id' not in df.columns or 'has_cactus' not in df.columns:    raise RuntimeError('Expected columns id and has_cactus not found in train.csv.')df = df.dropna(subset=['id', 'has_cactus']).copy()df['id'] = df['id'].astype(str)df['has_cactus'] = df['has_cactus'].astype(int)if set(df['has_cactus'].unique()) - {0, 1}:    raise RuntimeError('Unexpected class values in has_cactus column.')df['image_path'] = df['id'].apply(lambda x: str(train_img_dir / x))missing_images = int((~df['image_path'].apply(lambda p: Path(p).exists())).sum())if missing_images > 0:    raise RuntimeError(f'Missing train images referenced by csv: {missing_images}')sample_paths = df['image_path'].sample(n=min(300, len(df)), random_state=SEED).tolist()corrupt_count = 0for p in sample_paths:    try:        with Image.open(p) as img:            img.verify()    except Exception:        corrupt_count += 1if corrupt_count > 0:    raise RuntimeError(f'Corrupted train images found in sample check: {corrupt_count}')train_df, val_df = train_test_split(df, test_size=0.20, random_state=SEED, stratify=df['has_cactus'])if len(train_df) == 0 or len(val_df) == 0:    raise RuntimeError('Split creation failed: train or val is empty.')print(f'Total rows: {len(df)}')print(f'Train split: {len(train_df)}')print(f'Val split: {len(val_df)}')print(f'Test images: {len(list(test_img_dir.glob(''*.jpg'')))}')display(df.head(8))

In [ ]:
for folder in ['train', 'val']:    for cls in ['no_cactus', 'has_cactus']:        (DATA_DIR / folder / cls).mkdir(parents=True, exist_ok=True)for _, row in train_df.iterrows():    src = Path(row['image_path'])    cls_name = 'has_cactus' if int(row['has_cactus']) == 1 else 'no_cactus'    dst = DATA_DIR / 'train' / cls_name / src.name    if not dst.exists():        shutil.copy2(src, dst)for _, row in val_df.iterrows():    src = Path(row['image_path'])    cls_name = 'has_cactus' if int(row['has_cactus']) == 1 else 'no_cactus'    dst = DATA_DIR / 'val' / cls_name / src.name    if not dst.exists():        shutil.copy2(src, dst)cls_counts = {    'train_no_cactus': len(list((DATA_DIR / 'train' / 'no_cactus').glob('*.jpg'))),    'train_has_cactus': len(list((DATA_DIR / 'train' / 'has_cactus').glob('*.jpg'))),    'val_no_cactus': len(list((DATA_DIR / 'val' / 'no_cactus').glob('*.jpg'))),    'val_has_cactus': len(list((DATA_DIR / 'val' / 'has_cactus').glob('*.jpg')))}print(cls_counts)

In [ ]:
sample_df = train_df.sample(n=min(9, len(train_df)), random_state=SEED).reset_index(drop=True)fig, axes = plt.subplots(3, 3, figsize=(10, 10))for i in range(len(sample_df)):    row = sample_df.iloc[i]    img = Image.open(row['image_path']).convert('RGB')    axes.flatten()[i].imshow(img)    axes.flatten()[i].set_title('has_cactus=' + str(int(row['has_cactus'])))    axes.flatten()[i].axis('off')for j in range(len(sample_df), 9):    axes.flatten()[j].axis('off')plt.tight_layout()sample_grid = ARTIFACTS_DIR / 'sample_grid.png'plt.savefig(sample_grid, dpi=140)plt.show()

## YOLO26m-cls Training and Validation

In [ ]:
from ultralytics import YOLOmodel_candidates = ['yolo26m-cls.pt', 'yolo11m-cls.pt', 'yolov8m-cls.pt']model = Noneselected_weights = Nonefor w in model_candidates:    try:        model = YOLO(w)        selected_weights = w        break    except Exception:        continueif selected_weights is None:    raise RuntimeError('Could not load any classification weights for YOLO training.')print(f'Selected weights: {selected_weights}')train_result = model.train(    data=str(DATA_DIR),    epochs=3,    imgsz=128,    batch=64,    project=str(ARTIFACTS_DIR / 'yolo_runs'),    name='aerial_cactus_cls',    seed=SEED,    verbose=False)best_model_path = Path(model.trainer.best)print(f'Best model path: {best_model_path}')

In [ ]:
best_model = YOLO(str(best_model_path))val_paths = val_df['image_path'].tolist()y_true = val_df['has_cactus'].astype(int).tolist()y_pred = []scores = []for p in val_paths:    pred = best_model.predict(source=p, verbose=False)[0]    probs = pred.probs.data.cpu().numpy()    pred_idx = int(np.argmax(probs))    if hasattr(pred, 'names') and isinstance(pred.names, dict):        names = pred.names        if names.get(0, '').lower().startswith('has'):            mapped = 1 if pred_idx == 0 else 0        elif names.get(1, '').lower().startswith('has'):            mapped = 1 if pred_idx == 1 else 0        else:            mapped = pred_idx    else:        mapped = pred_idx    y_pred.append(int(mapped))    scores.append(float(np.max(probs)))val_acc = accuracy_score(y_true, y_pred)val_f1 = f1_score(y_true, y_pred)cm = confusion_matrix(y_true, y_pred)print(f'Validation accuracy: {val_acc:.4f}')print(f'Validation F1: {val_f1:.4f}')print('Classification report:')print(classification_report(y_true, y_pred, target_names=['no_cactus', 'has_cactus']))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))im = ax.imshow(cm, cmap='Blues')ax.set_title('Validation Confusion Matrix')ax.set_xlabel('Predicted')ax.set_ylabel('True')ax.set_xticks([0, 1])ax.set_yticks([0, 1])ax.set_xticklabels(['no_cactus', 'has_cactus'])ax.set_yticklabels(['no_cactus', 'has_cactus'])for i in range(2):    for j in range(2):        ax.text(j, i, str(cm[i, j]), ha='center', va='center', color='black')plt.tight_layout()cm_png = ARTIFACTS_DIR / 'confusion_matrix.png'plt.savefig(cm_png, dpi=140)plt.show()

## Honest Qualitative Analysis

In [ ]:
qual_df = val_df.copy().reset_index(drop=True)qual_df['pred'] = y_predqual_df['score'] = scoresqual_df['correct'] = (qual_df['has_cactus'].astype(int) == qual_df['pred'].astype(int)).astype(int)hard_cases = qual_df[qual_df['correct'] == 0].head(6)if len(hard_cases) == 0:    hard_cases = qual_df.sample(n=min(6, len(qual_df)), random_state=SEED)fig, axes = plt.subplots(2, 3, figsize=(12, 8))for i in range(len(hard_cases)):    row = hard_cases.iloc[i]    img = Image.open(row['image_path']).convert('RGB')    axes.flatten()[i].imshow(img)    axes.flatten()[i].set_title(f't={int(row["has_cactus"])} p={int(row["pred"])} s={row["score"]:.2f}')    axes.flatten()[i].axis('off')for j in range(len(hard_cases), 6):    axes.flatten()[j].axis('off')plt.tight_layout()qual_png = ARTIFACTS_DIR / 'qualitative_cases.png'plt.savefig(qual_png, dpi=140)plt.show()

## Save Real Outputs

In [ ]:
pred_df = pd.DataFrame({    'image_path': val_paths,    'y_true': y_true,    'y_pred': y_pred,    'score': scores})pred_csv = ARTIFACTS_DIR / 'val_predictions.csv'pred_df.to_csv(pred_csv, index=False)metrics = {    'dataset': 'kaggle://aerial-cactus-identification',    'rows_total': int(len(df)),    'split_counts': {        'train': int(len(train_df)),        'val': int(len(val_df))    },    'weights_selected': selected_weights,    'best_model_path': str(best_model_path),    'val_accuracy': float(val_acc),    'val_f1': float(val_f1)}metrics_json = ARTIFACTS_DIR / 'metrics.json'with open(metrics_json, 'w', encoding='utf-8') as f:    json.dump(metrics, f, indent=2)manifest = {    'competition_url': 'https://www.kaggle.com/competitions/aerial-cactus-identification/data',    'raw_dir': str(RAW_DIR),    'prepared_data_dir': str(DATA_DIR),    'metrics_json': str(metrics_json),    'pred_csv': str(pred_csv),    'sample_grid_png': str(sample_grid),    'confusion_matrix_png': str(cm_png),    'qualitative_png': str(qual_png)}manifest_json = ARTIFACTS_DIR / 'artifact_manifest.json'with open(manifest_json, 'w', encoding='utf-8') as f:    json.dump(manifest, f, indent=2)print('Saved outputs:')print(f'- {metrics_json}')print(f'- {pred_csv}')print(f'- {manifest_json}')print(f'- {sample_grid}')print(f'- {cm_png}')print(f'- {qual_png}')

## Limitations and Improvements- This notebook focuses on classification for Kaggle cactus labels; it does not include segmentation or object detection outputs because ground truth is image-level.- For stronger performance, increase epochs, tune augmentation, and use larger image size after validating resource limits.- For production deployment, add calibration and threshold tuning for precision/recall trade-offs.